# Light-VQSP-Net: Complete End-to-End Implementation

**Lightweight Variational Quantum-Assisted Scheduling and Phase Optimization Network**

This notebook implements an executable simulation of the methodology supplied in Section 3:

1. RIS-assisted downlink system and channel generation.
2. Bernoulli latest-packet arrivals, queue state, decoding and Age of Information (AoI).
3. Five-qubit variational quantum policy for age-aware stream/subchannel scheduling.
4. Three-layer QAOA-inspired Ising optimizer for RIS phase selection, with spatial entanglement and bounded continuous refinement.
5. Bilevel interaction: the scheduler proposes allocations, the RIS optimizer returns phases/SNR, and the resulting delivery outcomes update AoI and the policy.
6. Baseline comparisons, metrics, plots and result export.

**Important modeling note:** Some mathematical expressions and symbols in the supplied manuscript were lost during text extraction. The implementation below uses explicit, dimensionally consistent equations matching the accompanying descriptions. The channel simulator generates stochastic experimental scenarios; it does not claim to load measured field data. The statevector circuits are exact numerical simulations (not execution on quantum hardware).

## 1. Environment and reproducibility

In [ ]:
import os, time, math, random, json
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 2. Configuration

In [ ]:
@dataclass
class Config:
    # Network
    K: int = 6                 # traffic streams / destinations
    M: int = 3                 # orthogonal subchannels
    Nr: int = 2                # RIS rows
    Nc: int = 4                # RIS columns; total RIS elements = 8
    T: int = 80                # slots per episode
    EPISODES: int = 12
    max_tx_power: float = 1.0
    noise_power: float = 1e-8
    snr_threshold_db: float = 5.0
    arrival_prob: tuple = (0.75, 0.65, 0.60, 0.55, 0.45, 0.40)

    # Channel / geometry
    carrier_hz: float = 3.5e9
    wavelength: float = 3e8 / 3.5e9
    element_spacing_lambda: float = 0.5
    bs_ris_distance: float = 35.0
    ris_user_distances: tuple = (18., 22., 26., 30., 34., 38.)
    direct_distances: tuple = (45., 50., 55., 60., 65., 70.)
    pathloss_exp: float = 2.2
    rician_K: float = 5.0
    blockage_prob: float = 0.25
    blocked_amplitude: float = 0.08
    channel_estimation_noise: float = 0.04

    # QEA-TS
    policy_qubits: int = 5
    variational_layers: int = 3
    inverse_temperature: float = 4.0
    urgency_bonus: float = 0.35
    outage_penalty: float = 1.20
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_epsilon: float = 0.20
    actor_lr: float = 0.012
    critic_lr: float = 0.015

    # QARPO
    qaoa_depth: int = 3
    coordination_strength: float = 0.18
    max_refinement_step: float = 0.05
    phase_levels: int = 2       # binary phase encoding: 0 or pi
    qaoa_cost_scale: float = 0.25
    qaoa_mixer_scale: float = 0.35

    # Evaluation
    evaluation_episodes: int = 8

CFG = Config()
N_RIS = CFG.Nr * CFG.Nc
assert len(CFG.arrival_prob) == CFG.K
assert len(CFG.ris_user_distances) == CFG.K
print(CFG)
print("RIS elements:", N_RIS)


## 3. Physical system model and channel generation

In [ ]:
def cnormal(shape, rng=np.random):
    return (rng.normal(size=shape) + 1j*rng.normal(size=shape)) / np.sqrt(2.0)

def steering_upa(nr, nc, azimuth, elevation, spacing_lambda=0.5):
    """Normalized UPA steering vector using half-wavelength grid by default."""
    vals = []
    for r in range(nr):
        for c in range(nc):
            phase = 2*np.pi*spacing_lambda * (
                c*np.sin(azimuth)*np.cos(elevation) +
                r*np.sin(elevation)
            )
            vals.append(np.exp(1j*phase))
    return np.asarray(vals, dtype=np.complex128) / np.sqrt(nr*nc)

def pathloss(distance, exponent):
    # Reference path-loss gain at 1 m; normalized for stable simulation.
    return max(distance, 1.0) ** (-exponent)

def rician_channel(distance, nr, nc, K_factor, az, el, rng):
    los = steering_upa(nr, nc, az, el, CFG.element_spacing_lambda)
    nlos = cnormal(nr*nc, rng)
    beta = pathloss(distance, CFG.pathloss_exp)
    k_lin = 10**(K_factor/10)
    return np.sqrt(beta) * (
        np.sqrt(k_lin/(k_lin+1))*los +
        np.sqrt(1/(k_lin+1))*nlos
    )

def generate_channel_realization(rng=np.random):
    """Generate BS-RIS, RIS-user and blockage-prone direct channels for all streams/subchannels."""
    H_br = np.zeros((CFG.M, N_RIS), dtype=np.complex128)
    for m in range(CFG.M):
        H_br[m] = rician_channel(
            CFG.bs_ris_distance, CFG.Nr, CFG.Nc, CFG.rician_K,
            az=0.25 + 0.15*m, el=0.18, rng=rng
        )

    H_ru = np.zeros((CFG.K, CFG.M, N_RIS), dtype=np.complex128)
    h_d = np.zeros((CFG.K, CFG.M), dtype=np.complex128)
    blockage = np.zeros((CFG.K, CFG.M), dtype=np.int32)

    for k in range(CFG.K):
        for m in range(CFG.M):
            H_ru[k, m] = rician_channel(
                CFG.ris_user_distances[k], CFG.Nr, CFG.Nc, CFG.rician_K,
                az=0.35 + 0.08*k, el=0.12 + 0.02*m, rng=rng
            )
            d = CFG.direct_distances[k]
            direct = np.sqrt(pathloss(d, CFG.pathloss_exp)) * cnormal(1, rng)[0]
            blocked = int(rng.random() < CFG.blockage_prob)
            blockage[k, m] = blocked
            h_d[k, m] = direct * (CFG.blocked_amplitude if blocked else 1.0)

    # Imperfect/denoised estimates: additive estimation error is retained in the model.
    H_br_hat = H_br + CFG.channel_estimation_noise * cnormal(H_br.shape, rng)
    H_ru_hat = H_ru + CFG.channel_estimation_noise * cnormal(H_ru.shape, rng)
    h_d_hat = h_d + CFG.channel_estimation_noise * cnormal(h_d.shape, rng)
    return H_br, H_ru, h_d, H_br_hat, H_ru_hat, h_d_hat, blockage

def effective_channel(H_br_hat, H_ru_hat, h_d_hat, k, m, theta):
    v = np.exp(1j*np.asarray(theta))
    # Single-antenna BS/RIS-user links; cascaded RIS path plus direct path.
    return h_d_hat[k, m] + np.sum(H_ru_hat[k, m] * H_br_hat[m] * v)

def compute_snr_matrix(H_br_hat, H_ru_hat, h_d_hat, theta):
    h_eff = np.zeros((CFG.K, CFG.M), dtype=np.complex128)
    for k in range(CFG.K):
        for m in range(CFG.M):
            h_eff[k, m] = effective_channel(
                H_br_hat, H_ru_hat, h_d_hat, k, m, theta
            )
    snr = CFG.max_tx_power * np.abs(h_eff)**2 / CFG.noise_power
    return snr, h_eff

print("Channel model functions ready.")


## 4. Latest-packet buffers and Age of Information dynamics

In [ ]:
class AoIEnvironment:
    """One episode of Bernoulli arrivals, latest-packet buffers and RIS-assisted downlink."""
    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.age = np.ones(CFG.K, dtype=np.float64)
        self.packet_age = np.zeros(CFG.K, dtype=np.float64)
        self.buffer = np.zeros(CFG.K, dtype=np.int32)
        self.slot = 0
        self.last_arrivals = np.zeros(CFG.K, dtype=np.int32)
        self.last_delivery = np.zeros(CFG.K, dtype=np.int32)
        self.channels = generate_channel_realization(self.rng)
        return self.get_state()

    def get_state(self):
        return {
            "age": self.age.copy(),
            "packet_age": self.packet_age.copy(),
            "buffer": self.buffer.copy(),
            "slot": self.slot
        }

    def step(self, allocation, theta):
        """
        allocation: list of (stream k, subchannel m), with unique streams/subchannels.
        theta: RIS phase vector in radians.
        Slot order: packet arrivals/replacement -> transmission -> age update.
        """
        H_br, H_ru, h_d, H_br_hat, H_ru_hat, h_d_hat, blockage = self.channels

        arrivals = (self.rng.random(CFG.K) < np.asarray(CFG.arrival_prob)).astype(np.int32)
        # New arrivals replace old packets and have system time 1 at this slot.
        self.packet_age = np.where(arrivals == 1, 1.0, self.packet_age + (self.buffer > 0))
        self.buffer = np.where(arrivals == 1, 1, self.buffer)
        self.last_arrivals = arrivals

        snr_matrix, _ = compute_snr_matrix(H_br_hat, H_ru_hat, h_d_hat, theta)
        threshold = 10**(CFG.snr_threshold_db/10)
        delivery = np.zeros(CFG.K, dtype=np.int32)
        used_streams, used_subchannels = set(), set()

        for k, m in allocation:
            if k in used_streams or m in used_subchannels:
                continue
            used_streams.add(k); used_subchannels.add(m)
            if self.buffer[k] == 1 and snr_matrix[k, m] >= threshold:
                delivery[k] = 1

        # Successful delivery resets destination age to the delivered packet's system time.
        # Otherwise destination age increases by one slot.
        self.age = np.where(delivery == 1, np.maximum(self.packet_age, 1.0), self.age + 1.0)
        self.buffer = np.where(delivery == 1, 0, self.buffer)
        self.packet_age = np.where(delivery == 1, 0.0, self.packet_age)
        self.last_delivery = delivery
        self.slot += 1

        # Age-adaptive reward: negative sum-AoI plus urgency bonus and outage penalty.
        urgency = np.sum(np.log1p(self.age) * self.buffer)
        outage = sum(1 for k, m in allocation if snr_matrix[k, m] < threshold)
        reward = -float(np.sum(self.age)) + CFG.urgency_bonus*urgency - CFG.outage_penalty*outage
        info = {
            "snr_matrix": snr_matrix,
            "delivery": delivery.copy(),
            "arrivals": arrivals.copy(),
            "blockage": blockage.copy(),
            "sum_aoi": float(np.sum(self.age)),
            "success_rate": float(np.mean(delivery)),
            "mean_snr_db": float(10*np.log10(np.maximum(snr_matrix, 1e-15)).mean())
        }
        return self.get_state(), reward, info

print("AoI environment ready.")


## 5. Quantum statevector utilities

In [ ]:
# Exact statevector simulation for small circuits.
def apply_single_qubit(state, gate, target, n_qubits):
    tensor = state.reshape([2]*n_qubits)
    tensor = np.moveaxis(tensor, target, 0)
    tensor = np.tensordot(gate, tensor, axes=([1], [0]))
    tensor = np.moveaxis(tensor, 0, target)
    return tensor.reshape(-1)

def apply_cnot(state, control, target, n_qubits):
    out = state.copy()
    for idx in range(2**n_qubits):
        if ((idx >> control) & 1) and not ((idx >> target) & 1):
            j = idx | (1 << target)
            out[idx], out[j] = state[j], state[idx]
    return out

def RX(a):
    return np.array([[np.cos(a/2), -1j*np.sin(a/2)],
                     [-1j*np.sin(a/2), np.cos(a/2)]], dtype=np.complex128)

def RY(a):
    return np.array([[np.cos(a/2), -np.sin(a/2)],
                     [np.sin(a/2), np.cos(a/2)]], dtype=np.complex128)

def RZ(a):
    return np.array([[np.exp(-1j*a/2), 0],
                     [0, np.exp(1j*a/2)]], dtype=np.complex128)

def hadamard_state(n):
    s = np.zeros(2**n, dtype=np.complex128); s[0] = 1.0
    H = np.array([[1,1],[1,-1]], dtype=np.complex128)/np.sqrt(2)
    for q in range(n):
        s = apply_single_qubit(s, H, q, n)
    return s

def normalize_probs(state):
    p = np.abs(state)**2
    return p / max(p.sum(), 1e-15)

print("Statevector utilities ready.")


## 6. QEA-TS: five-qubit variational quantum policy network

In [ ]:
class VariationalQuantumScheduler:
    """
    5-qubit feature-encoded ring-entangled policy.
    3 layers x 5 qubits x 3 Euler angles = 45 trainable angles.
    Pair (stream, subchannel) maps to a computational basis outcome.
    """
    def __init__(self, n_qubits=5, layers=3, seed=42):
        self.n = n_qubits
        self.layers = layers
        self.rng = np.random.default_rng(seed)
        self.params = self.rng.normal(0, 0.08, size=(layers, n_qubits, 3))
        self.old_params = self.params.copy()
        self.critic_w = np.zeros(2*CFG.K + 2, dtype=np.float64)
        self.critic_b = 0.0

    def encode(self, state_features):
        s = hadamard_state(self.n)
        x = np.resize(np.asarray(state_features, dtype=float), self.n*2)
        for q in range(self.n):
            s = apply_single_qubit(s, RY(float(np.pi*np.tanh(x[2*q]))), q, self.n)
            s = apply_single_qubit(s, RZ(float(np.pi*np.tanh(x[2*q+1]))), q, self.n)
        return s

    def circuit(self, state_features, params=None):
        p = self.params if params is None else params
        s = self.encode(state_features)
        for l in range(self.layers):
            for q in range(self.n):
                s = apply_single_qubit(s, RY(p[l,q,0]), q, self.n)
                s = apply_single_qubit(s, RZ(p[l,q,1]), q, self.n)
                s = apply_single_qubit(s, RX(p[l,q,2]), q, self.n)
            for q in range(self.n):
                s = apply_cnot(s, q, (q+1) % self.n, self.n)
        return s

    def state_features(self, env_state, snr_matrix):
        age = env_state["age"]
        buf = env_state["buffer"].astype(float)
        # Staleness-weighted telemetry plus normalized channel descriptor.
        channel = np.max(snr_matrix, axis=1)
        channel = channel / (np.max(channel) + 1e-12)
        urgency = np.tanh(age / (np.mean(age) + 1e-8))
        return np.concatenate([urgency, buf, channel, [np.mean(age)]])

    def pair_probability(self, probs, k, m):
        pair_index = k*CFG.M + m
        return float(probs[pair_index % len(probs)])

    def act(self, env_state, snr_matrix, temperature=None):
        temperature = CFG.inverse_temperature if temperature is None else temperature
        features = self.state_features(env_state, snr_matrix)
        probs = normalize_probs(self.circuit(features))
        available_streams = set(np.where(env_state["buffer"] > 0)[0].tolist())
        used_streams, used_subchannels, allocation = set(), set(), []
        selected_probs = []
        for m in range(CFG.M):
            candidates = [k for k in available_streams if k not in used_streams]
            if not candidates:
                break
            logits = np.array([
                np.log(self.pair_probability(probs, k, m) + 1e-12)
                + np.log1p(env_state["age"][k])
                for k in candidates
            ])
            logits = temperature * (logits - logits.max())
            weights = np.exp(np.clip(logits, -50, 50))
            weights /= weights.sum()
            k = int(self.rng.choice(candidates, p=weights))
            allocation.append((k, m))
            used_streams.add(k); used_subchannels.add(m)
            selected_probs.append((k, m, max(self.pair_probability(probs,k,m), 1e-12)))
        return allocation, selected_probs, features

    def value(self, features):
        x = np.resize(np.asarray(features, dtype=float), self.critic_w.size)
        return float(np.dot(self.critic_w, x) + self.critic_b)

    def update_critic(self, feature_batch, target_batch):
        X = np.asarray([np.resize(x, self.critic_w.size) for x in feature_batch])
        y = np.asarray(target_batch, dtype=float)
        pred = X @ self.critic_w + self.critic_b
        err = pred - y
        self.critic_w -= CFG.critic_lr * (2*X.T@err/max(len(y),1))
        self.critic_b -= CFG.critic_lr * 2*float(np.mean(err))
        return float(np.mean(err**2))

    def parameter_shift_logprob_grad(self, features, selected_pairs):
        """Exact parameter-shift derivative of log measured pair probability."""
        grad = np.zeros_like(self.params)
        if not selected_pairs:
            return grad
        eps = np.pi/2
        base_params = self.params.copy()
        base_probs = normalize_probs(self.circuit(features, base_params))
        for idx in np.ndindex(self.params.shape):
            plus = base_params.copy(); minus = base_params.copy()
            plus[idx] += eps; minus[idx] -= eps
            pp = normalize_probs(self.circuit(features, plus))
            pm = normalize_probs(self.circuit(features, minus))
            g = 0.0
            for k, m, _ in selected_pairs:
                a = self.pair_probability(pp,k,m)
                b = self.pair_probability(pm,k,m)
                g += 0.5*(a-b)/(self.pair_probability(base_probs,k,m)+1e-12)
            grad[idx] = g / len(selected_pairs)
        self.params = base_params
        return grad

    def policy_update(self, episode_records):
        if not episode_records:
            return 0.0
        rewards = np.array([r["reward"] for r in episode_records], dtype=float)
        values = np.array([r["value"] for r in episode_records], dtype=float)
        returns = np.zeros_like(rewards)
        running = 0.0
        for t in reversed(range(len(rewards))):
            running = rewards[t] + CFG.gamma*running
            returns[t] = running
        advantages = returns - values
        advantages = (advantages - advantages.mean())/(advantages.std()+1e-8)
        total_norm = 0.0
        for rec, adv in zip(episode_records, advantages):
            if not rec["pairs"]:
                continue
            grad = self.parameter_shift_logprob_grad(rec["features"], rec["pairs"])
            # Clipped-policy-inspired bounded actor step; prevents unstable angle jumps.
            grad = np.clip(grad * adv, -1.0, 1.0)
            self.params += CFG.actor_lr * grad
            self.params = (self.params + np.pi) % (2*np.pi) - np.pi
            total_norm += float(np.linalg.norm(grad))
        self.update_critic([r["features"] for r in episode_records], returns)
        return total_norm/max(len(episode_records),1)

scheduler = VariationalQuantumScheduler(CFG.policy_qubits, CFG.variational_layers, SEED)
print("Policy parameters:", scheduler.params.size, "(expected 45)")


## 7. QARPO: Ising mapping, QAOA and continuous RIS phase refinement

In [ ]:
def build_ris_quadratic(H_br_hat, H_ru_hat, h_d_hat, allocation):
    """Construct aggregate |d + a^T v|^2 = v^H Q v + 2 Re(v^H b) + const."""
    Q = np.zeros((N_RIS, N_RIS), dtype=np.complex128)
    b = np.zeros(N_RIS, dtype=np.complex128)
    for k, m in allocation:
        a = H_ru_hat[k,m] * H_br_hat[m]
        d = h_d_hat[k,m]
        Q += np.outer(np.conj(a), a)
        b += np.conj(a) * d
    return Q, b

def quadratic_snr_objective(theta, Q, b):
    v = np.exp(1j*np.asarray(theta))
    return float(np.real(np.vdot(v, Q@v) + 2*np.vdot(v,b)))

def map_to_ising(Q, b):
    """Binary phase restriction v_i=s_i in {+1,-1}; return J and h for max s^T J s + 2h^T s."""
    Qr = np.real(Q)
    J = np.zeros_like(Qr)
    for i in range(N_RIS):
        for j in range(i+1, N_RIS):
            J[i,j] = J[j,i] = 2.0*Qr[i,j]
    h = 2.0*np.real(b)
    scale = max(np.max(np.abs(J)), np.max(np.abs(h)), 1e-9)
    return J/scale, h/scale, scale

def apply_zz_cost(state, gamma, J, h, n):
    out = state.copy()
    for idx in range(2**n):
        z = np.array([1 if ((idx>>q)&1)==0 else -1 for q in range(n)])
        energy = sum(J[i,j]*z[i]*z[j] for i in range(n) for j in range(i+1,n))
        energy += np.dot(h,z)
        out[idx] *= np.exp(-1j*gamma*energy)
    return out

def apply_x_mixer(state, beta, n):
    out = state.copy()
    for q in range(n):
        out = apply_single_qubit(out, RX(2*beta), q, n)
    return out

def apply_xx_coordination(state, strength, n):
    # exp(-i strength X_i X_j), implemented as H-H, ZZ, H-H.
    H = np.array([[1,1],[1,-1]], dtype=np.complex128)/np.sqrt(2)
    out = state.copy()
    for q in range(n-1):
        out = apply_single_qubit(out, H, q, n)
        out = apply_single_qubit(out, H, q+1, n)
        # ZZ rotation: diagonal exp(-i strength*z_i*z_j)
        for idx in range(2**n):
            zi = 1 if ((idx>>q)&1)==0 else -1
            zj = 1 if ((idx>>(q+1))&1)==0 else -1
            out[idx] *= np.exp(-1j*strength*zi*zj)
        out = apply_single_qubit(out, H, q, n)
        out = apply_single_qubit(out, H, q+1, n)
    return out

def qaoa_ris_optimizer(H_br_hat, H_ru_hat, h_d_hat, allocation, depth=3):
    if len(allocation) == 0:
        return np.zeros(N_RIS), np.eye(N_RIS, dtype=np.complex128), np.zeros((CFG.K,CFG.M))
    Q, b = build_ris_quadratic(H_br_hat, H_ru_hat, h_d_hat, allocation)
    J, h, scale = map_to_ising(Q, b)
    state = hadamard_state(N_RIS)
    # Deterministic linear-ramp angles; no classical inner-loop search.
    for layer in range(depth):
        gamma = CFG.qaoa_cost_scale * (layer+1)/depth
        beta = CFG.qaoa_mixer_scale * (depth-layer)/depth
        state = apply_zz_cost(state, gamma, J, h, N_RIS)
        state = apply_x_mixer(state, beta, N_RIS)
    state = apply_xx_coordination(state, CFG.coordination_strength, N_RIS)

    # Single-qubit Z expectations -> coarse binary phases.
    probs = normalize_probs(state)
    zexp = np.zeros(N_RIS)
    for q in range(N_RIS):
        for idx,p in enumerate(probs):
            zexp[q] += p*(1 if ((idx>>q)&1)==0 else -1)
    spins = np.where(zexp >= 0, 1.0, -1.0)
    theta = np.where(spins > 0, 0.0, np.pi)

    # One bounded gradient-ascent refinement step on the continuous phase objective.
    v = np.exp(1j*theta)
    grad = 2*np.imag(np.conj(v) * (Q@v + b))
    theta = theta + np.clip(0.02*grad/(np.max(np.abs(grad))+1e-12),
                            -CFG.max_refinement_step, CFG.max_refinement_step)
    theta = np.mod(theta, 2*np.pi)
    Phi = np.diag(np.exp(1j*theta))
    return theta, Phi, Q

print("QAOA RIS optimizer ready; QAOA qubits:", N_RIS)


## 8. Training loop: bilevel scheduler + RIS optimizer + AoI feedback

In [ ]:
def run_training(cfg=CFG, seed=SEED):
    global scheduler
    scheduler = VariationalQuantumScheduler(cfg.policy_qubits, cfg.variational_layers, seed)
    history = []
    start_all = time.time()

    for ep in range(cfg.EPISODES):
        env = AoIEnvironment(seed + 1000 + ep)
        state = env.reset()
        records = []
        ep_ages, ep_rewards, ep_success, ep_snr = [], [], [], []

        for t in range(cfg.T):
            # Channel descriptor from the current denoised estimates.
            H_br, H_ru, h_d, H_br_hat, H_ru_hat, h_d_hat, blockage = env.channels
            theta_probe = np.zeros(N_RIS)
            snr_probe, _ = compute_snr_matrix(H_br_hat,H_ru_hat,h_d_hat,theta_probe)
            features = scheduler.state_features(state, snr_probe)
            value = scheduler.value(features)

            allocation, selected_pairs, features = scheduler.act(state, snr_probe)

            # Inner optimization returns RIS phases for the scheduled stream set.
            theta, Phi, Q = qaoa_ris_optimizer(
                H_br_hat,H_ru_hat,h_d_hat,allocation,cfg.qaoa_depth
            )
            next_state, reward, info = env.step(allocation, theta)

            records.append({
                "features": features,
                "pairs": selected_pairs,
                "reward": reward,
                "value": value
            })
            ep_ages.append(info["sum_aoi"])
            ep_rewards.append(reward)
            ep_success.append(info["success_rate"])
            ep_snr.append(info["mean_snr_db"])
            state = next_state

            # Independent small-scale fading realization each slot.
            env.channels = generate_channel_realization(env.rng)

        grad_norm = scheduler.policy_update(records)
        history.append({
            "episode": ep+1,
            "mean_sum_aoi": float(np.mean(ep_ages)),
            "mean_reward": float(np.mean(ep_rewards)),
            "delivery_rate": float(np.mean(ep_success)),
            "mean_snr_db": float(np.mean(ep_snr)),
            "actor_grad_norm": grad_norm
        })
        print(f"Episode {ep+1:02d}/{cfg.EPISODES} | "
              f"Mean sum-AoI={np.mean(ep_ages):.3f} | "
              f"Reward={np.mean(ep_rewards):.3f} | "
              f"Delivery={np.mean(ep_success):.3f}")

    print(f"Training completed in {time.time()-start_all:.1f} seconds.")
    return pd.DataFrame(history), scheduler

training_history, trained_scheduler = run_training()
training_history


## 9. Evaluation policies and baseline methods

In [ ]:
def evaluate_policy(policy_name, episodes=CFG.evaluation_episodes, seed_offset=50000):
    rows = []
    for ep in range(episodes):
        env = AoIEnvironment(seed_offset+ep)
        state = env.reset()
        for t in range(CFG.T):
            H_br, H_ru, h_d, H_br_hat, H_ru_hat, h_d_hat, blockage = env.channels
            theta0 = np.zeros(N_RIS)
            snr0, _ = compute_snr_matrix(H_br_hat,H_ru_hat,h_d_hat,theta0)

            if policy_name == "Light-VQSP-Net":
                allocation, _, _ = trained_scheduler.act(state, snr0)
                theta, _, _ = qaoa_ris_optimizer(H_br_hat,H_ru_hat,h_d_hat,allocation,CFG.qaoa_depth)
            elif policy_name == "Age-greedy + QAOA":
                candidates = sorted(
                    np.where(state["buffer"] > 0)[0],
                    key=lambda k: state["age"][k], reverse=True
                )
                allocation = [(int(k),m) for m,k in enumerate(candidates[:CFG.M])]
                theta, _, _ = qaoa_ris_optimizer(H_br_hat,H_ru_hat,h_d_hat,allocation,CFG.qaoa_depth)
            elif policy_name == "Random + QAOA":
                candidates = np.where(state["buffer"] > 0)[0].tolist()
                env.rng.shuffle(candidates)
                allocation = [(int(k),m) for m,k in enumerate(candidates[:CFG.M])]
                theta, _, _ = qaoa_ris_optimizer(H_br_hat,H_ru_hat,h_d_hat,allocation,CFG.qaoa_depth)
            elif policy_name == "Age-greedy + random RIS":
                candidates = sorted(
                    np.where(state["buffer"] > 0)[0],
                    key=lambda k: state["age"][k], reverse=True
                )
                allocation = [(int(k),m) for m,k in enumerate(candidates[:CFG.M])]
                theta = env.rng.uniform(0,2*np.pi,N_RIS)
            else:
                raise ValueError("Unknown policy: "+policy_name)

            state, reward, info = env.step(allocation,theta)
            rows.append({
                "policy": policy_name,
                "episode": ep,
                "slot": t,
                "sum_aoi": info["sum_aoi"],
                "delivery_rate": info["success_rate"],
                "mean_snr_db": info["mean_snr_db"],
                "reward": reward,
                "scheduled_streams": len(allocation)
            })
            env.channels = generate_channel_realization(env.rng)
    return pd.DataFrame(rows)

policy_names = [
    "Light-VQSP-Net",
    "Age-greedy + QAOA",
    "Random + QAOA",
    "Age-greedy + random RIS"
]
evaluation_frames = [evaluate_policy(name) for name in policy_names]
evaluation_df = pd.concat(evaluation_frames, ignore_index=True)
summary = evaluation_df.groupby("policy").agg(
    Mean_Sum_AoI=("sum_aoi","mean"),
    Mean_Delivery_Rate=("delivery_rate","mean"),
    Mean_SNR_dB=("mean_snr_db","mean"),
    Mean_Reward=("reward","mean"),
    Mean_Scheduled=("scheduled_streams","mean")
).reset_index()
summary


## 10. Performance metrics

In [ ]:
def calculate_metrics(df):
    metrics = {}
    for policy, g in df.groupby("policy"):
        metrics[policy] = {
            "Average sum-AoI": g["sum_aoi"].mean(),
            "Median sum-AoI": g["sum_aoi"].median(),
            "95th percentile sum-AoI": g["sum_aoi"].quantile(0.95),
            "Mean delivery rate": g["delivery_rate"].mean(),
            "Mean SNR (dB)": g["mean_snr_db"].mean(),
            "Mean reward": g["reward"].mean(),
            "Mean scheduled streams/slot": g["scheduled_streams"].mean()
        }
    return pd.DataFrame(metrics).T

metrics_table = calculate_metrics(evaluation_df)
display(metrics_table.round(4))


## 11. Visualization

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(training_history["episode"], training_history["mean_sum_aoi"], marker="o")
plt.xlabel("Training episode"); plt.ylabel("Mean sum-AoI")
plt.title("Light-VQSP-Net training: Age of Information")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(9,5))
plt.plot(training_history["episode"], training_history["mean_reward"], marker="o")
plt.xlabel("Training episode"); plt.ylabel("Mean reward")
plt.title("Light-VQSP-Net training reward")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

plot_data = evaluation_df.groupby("policy")["sum_aoi"].mean().sort_values()
plt.figure(figsize=(10,5))
plot_data.plot(kind="bar")
plt.ylabel("Average sum-AoI"); plt.xlabel("Policy")
plt.title("Baseline comparison: average sum-AoI (lower is better)")
plt.xticks(rotation=20, ha="right"); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

plot_data = evaluation_df.groupby("policy")["delivery_rate"].mean()
plt.figure(figsize=(10,5))
plot_data.plot(kind="bar")
plt.ylabel("Mean delivery rate"); plt.xlabel("Policy")
plt.title("Successful update delivery rate")
plt.xticks(rotation=20, ha="right"); plt.ylim(0,1)
plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()


## 12. Single-slot diagnostic: scheduling, RIS phases and SNR

In [ ]:
diagnostic_env = AoIEnvironment(SEED+999)
diagnostic_state = diagnostic_env.reset()
H_br, H_ru, h_d, H_br_hat, H_ru_hat, h_d_hat, blockage = diagnostic_env.channels
snr_initial, _ = compute_snr_matrix(H_br_hat,H_ru_hat,h_d_hat,np.zeros(N_RIS))
diagnostic_allocation, diagnostic_pairs, diagnostic_features = trained_scheduler.act(
    diagnostic_state,snr_initial
)
diagnostic_theta, diagnostic_Phi, diagnostic_Q = qaoa_ris_optimizer(
    H_br_hat,H_ru_hat,h_d_hat,diagnostic_allocation,CFG.qaoa_depth
)
snr_optimized, _ = compute_snr_matrix(H_br_hat,H_ru_hat,h_d_hat,diagnostic_theta)

print("Initial buffer:", diagnostic_state["buffer"])
print("Initial AoI:", diagnostic_state["age"])
print("Scheduled (stream, subchannel) pairs:", diagnostic_allocation)
print("Optimized RIS phases (radians):", np.round(diagnostic_theta,4))
print("RIS reflection matrix shape:", diagnostic_Phi.shape)
print("Mean SNR before RIS optimization (dB):",
      round(float(10*np.log10(np.maximum(snr_initial,1e-15)).mean()),3))
print("Mean SNR after RIS optimization (dB):",
      round(float(10*np.log10(np.maximum(snr_optimized,1e-15)).mean()),3))


## 13. Export experimental results

In [ ]:
OUTPUT_DIR = "light_vqsp_net_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_history.to_csv(os.path.join(OUTPUT_DIR,"training_history.csv"),index=False)
evaluation_df.to_csv(os.path.join(OUTPUT_DIR,"slot_level_evaluation.csv"),index=False)
summary.to_csv(os.path.join(OUTPUT_DIR,"baseline_summary.csv"),index=False)
metrics_table.to_csv(os.path.join(OUTPUT_DIR,"performance_metrics.csv"))

config_dict = {k:v for k,v in CFG.__dict__.items()}
with open(os.path.join(OUTPUT_DIR,"configuration.json"),"w") as f:
    json.dump(config_dict,f,indent=2)

np.save(os.path.join(OUTPUT_DIR,"trained_policy_angles.npy"),trained_scheduler.params)
print("Saved outputs to:", os.path.abspath(OUTPUT_DIR))
print(os.listdir(OUTPUT_DIR))


## 14. Interpretation and experimental reporting

Report the following from the generated tables:

- **Average sum-AoI:** primary objective; lower is better.
- **95th percentile sum-AoI:** tail freshness performance.
- **Delivery rate:** fraction of destinations successfully updated per slot.
- **Mean SNR:** received-link quality under the selected RIS phases.
- **Mean reward:** shaped training objective.
- **Mean scheduled streams/slot:** utilization of the available subchannels.

### Reproducibility and limitations

- The channel/arrival process is stochastic and reproducible under the configured seed.
- Channel estimates are imperfect and are used by the scheduler and RIS optimizer.
- The RIS QAOA stage uses binary phase encoding (0 or π), followed by a bounded continuous refinement step.
- The policy circuit has 5 qubits and 45 trainable Euler-rotation angles.
- The critic is classical. The policy circuit is simulated with a NumPy statevector.
- The evaluation baselines are heuristic comparators, not published benchmark implementations.
- For paper-grade validation, repeat experiments over multiple independent seeds, report mean ± standard deviation and confidence intervals, and match all physical parameters (carrier frequency, geometry, transmit power, noise figure, path-loss model and RIS hardware constraints) to the target deployment.
- This simulation is not evidence of quantum advantage. A hardware-backed quantum experiment or a controlled complexity/runtime study would be required for such a claim.
